# GTA6 Clipper: Signals (Colab)

Finds **chat spikes** and **Most Replayed** peaks in the top GTA 6 streams and long videos,
and writes the windows to the `Moments` tab. YouTube blocks this on GitHub, so it runs here.
The Watcher (Gemini) checks these windows automatically on GitHub afterwards.

**Every time:** `Runtime → Run all`, then allow the Google sign-in pop-up (your own account).
Takes about 5–25 minutes. You can close the tab once the last cell prints `Moments: ...`.

**One-time setup** (key icon 🔑 on the left = *Secrets*):
- `GH_TOKEN`: a GitHub fine-grained token with **read-only Contents** access to `gta6-clipper`
  (github.com → Settings → Developer settings → Fine-grained tokens). Turn on *Notebook access*.
- `YOUTUBE_API_KEY` (optional): adds comment timestamps. GitHub already does those, so you can skip it.


In [ ]:
# 1. Get the latest code (fresh copy every run; the token is never printed or kept)
import base64, os, shutil, subprocess
from google.colab import userdata

REPO = "/content/gta6-clipper"
os.chdir("/content")  # never stand inside the folder we're about to delete
shutil.rmtree(REPO, ignore_errors=True)
token = userdata.get("GH_TOKEN").strip()
auth_header = "Authorization: Basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()
p = subprocess.run(["git", "-c", f"http.extraHeader={auth_header}", "clone", "-q", "--depth", "1",
                    "https://github.com/VYOM222444/gta6-clipper.git", REPO],
                   capture_output=True, text=True)
del token, auth_header
if p.returncode != 0:
    msg = p.stderr.strip().splitlines()[-1] if p.stderr.strip() else "unknown error"
    raise SystemExit("Could not download the code: " + msg +
                     "\nCheck the GH_TOKEN secret (still valid? Contents: Read-only on gta6-clipper? Notebook access on?)")
print("Code ready:", subprocess.run(["git", "-C", REPO, "log", "-1", "--format=%h %s"],
                                    capture_output=True, text=True).stdout)


In [ ]:
# 2. Install yt-dlp (+ sheet library)
!pip -q install -r /content/gta6-clipper/requirements-signals.txt


In [ ]:
# 3. Sign in with your Google account (for the sheet)
from google.colab import auth, drive
import os, shutil
auth.authenticate_user()
drive.mount("/content/drive")
# Optional: YouTube cookies (only if YouTube keeps saying "confirm you're not a bot").
# A copy is used because yt-dlp rewrites the file.
COOKIES_DRIVE = "/content/drive/MyDrive/GTA6 Clipper/cookies.txt"
if os.path.exists(COOKIES_DRIVE):
    shutil.copy(COOKIES_DRIVE, "/content/cookies.txt")
    os.environ["YTDLP_COOKIES"] = "/content/cookies.txt"
    print("Using YouTube cookies from Drive.")
print("Signed in.")


In [ ]:
# 4. Find moments
import os, sys
os.chdir(REPO)
sys.path.insert(0, REPO)
os.environ["SIGNALS_MODE"] = "full"
os.environ["YTDLP_JS_RUNTIME"] = "node"
try:
    os.environ["YOUTUBE_API_KEY"] = userdata.get("YOUTUBE_API_KEY")
except Exception:
    os.environ.pop("YOUTUBE_API_KEY", None)

from agents import signals
signals.main()
